# 01. 토스광고 CTR 데이터 메모리 안전형(Memory-Safe) 프로파일링 & EDA

이 노트북은 약 **9.4 GB** 규모의 원천 데이터(`train.parquet` ~8.2GB, `test.parquet` ~1.2GB)를 **OOM(Out-of-Memory) 없이** `pyarrow` 메타데이터와 `duckdb` 스트리밍 쿼리로 분석합니다.

### 핵심 확인 목표
1. **스키마 및 피처 그룹 해독**: 119개 전체 컬럼의 정확한 이름, 타입(`dtype`), 그룹별 개수 확인
2. **타겟(`clicked`) 및 시간축(`day_of_week`, `hour`) 분포**: 전체 CTR 및 `train` vs `test` 시간축 비교 → **Local Validation 전략 확정**
3. **결측치 및 카디널리티 프로파일링**: 범주형 vs 수치형 분류 및 다운캐스팅(`min`/`max`) 가능 범위 산출
4. **`seq` 시퀀스 컬럼 해부**: 문자열 포맷, 길이 분포, 디스크 용량 비중 파악
5. **경량 요약 리포트(`eda_summary.json`) 저장**: 로컬/다음 노트북(`02_data_diet_and_cv.ipynb`)과 호환되는 수십 KB 요약 산출물 생성

In [ ]:
from google.colab import drive
import json
import math
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import os as _os
if not _os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

# 프로젝트 저장소 경로가 Google Drive에 있다면 sys.path에 src를 추가해 toss_ctr 유틸을 연동합니다.
PROJECT_ROOT = Path('/content/drive/MyDrive/toss_ctr')
SRC_DIR = PROJECT_ROOT / 'src'
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# TODO: Google Drive 내 실제 데이터 폴더 경로로 수정하세요.
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
OUT_DIR = PROJECT_ROOT / 'reports'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.parquet'
TEST_PATH = DATA_DIR / 'test.parquet'
SUB_PATH = DATA_DIR / 'sample_submission.csv'

for p in [TRAIN_PATH, TEST_PATH, SUB_PATH]:
    size_gb = p.stat().st_size / (1024 ** 3) if p.exists() else -1
    print(f'{p.name}: exists={p.exists()}, size={size_gb:.2f} GB')

### [해석] 실행 환경 및 원천 파일 크기 점검
- `train.parquet`(~8.2GB), `test.parquet`(~1.2GB), `sample_submission.csv` 존재 여부(`exists=True`)를 먼저 확인합니다.
- 일반 `pandas.read_parquet()`로 일괄 로드하면 9GB 이상의 RAM 스파이크로 Colab 기본 런타임(12~15GB)이 즉시 OOM 종료될 수 있으므로, 이후 모든 셀은 **Parquet 푸터 메타데이터 조회** 및 **DuckDB 스트리밍 집계**만 사용합니다.

### [분석 1] Parquet 메타데이터 기반 스키마 및 컬럼별 압축 용량 비중
- 데이터를 RAM에 적재하지 않고 Parquet 푸터(Footer) 메타데이터만 읽어 **행 수, Row Group 구조, 컬럼별 디스크 점유 용량 Top 15**를 확인합니다.
- 특히 `seq` 컬럼이 8.2 GB 중 몇 %를 차지하는지 확인하면 정형 테이블 분리 시 줄어드는 용량을 즉시 계산할 수 있습니다.

In [ ]:
train_pf = pq.ParquetFile(TRAIN_PATH)
test_pf = pq.ParquetFile(TEST_PATH)

print(f'Train rows: {train_pf.metadata.num_rows:,}, cols: {train_pf.metadata.num_columns}, row_groups: {train_pf.num_row_groups}')
print(f'Test  rows: {test_pf.metadata.num_rows:,}, cols: {test_pf.metadata.num_columns}, row_groups: {test_pf.num_row_groups}')

col_sizes = {}
col_types = {}
meta = train_pf.metadata
for rg_idx in range(meta.num_row_groups):
    rg = meta.row_group(rg_idx)
    for col_idx in range(rg.num_columns):
        col = rg.column(col_idx)
        name = col.path_in_schema
        col_sizes[name] = col_sizes.get(name, 0) + col.total_compressed_size
        col_types[name] = col.physical_type

schema_df = pd.DataFrame([
    {
        'column': col_name,
        'arrow_type': str(train_pf.schema_arrow.field(col_name).type),
        'compressed_mb': col_sizes.get(col_name, 0) / (1024 ** 2),
    }
    for col_name in train_pf.schema_arrow.names
])
schema_df['size_share_pct'] = schema_df['compressed_mb'] / schema_df['compressed_mb'].sum() * 100
display(schema_df.sort_values('compressed_mb', ascending=False).head(15))

train_cols = set(train_pf.schema_arrow.names)
test_cols = set(test_pf.schema_arrow.names)
print('Train only:', train_cols - test_cols)
print('Test only :', test_cols - train_cols)

### [해석 1] 스키마 정합성 및 `seq` 분리 효과 근거
- `Train only`는 타겟 컬럼 `clicked`, `Test only`는 식별자 컬럼 `ID`로 나타나야 117개 입력 피처(`seq` 포함 118개)의 스키마 정합성이 확인됩니다.
- `seq` 컬럼이 전체 압축 용량의 절대 다수를 차지하는 경우, `02_data_diet_and_cv.ipynb`에서 **`seq`를 별도 파일(`train_seq.parquet`, `test_seq.parquet`)로 분리**하고 정형 피처를 `float32`/`int8~int32`로 다운캐스팅하면 GBDT 트랙 I/O 속도와 RAM 효율을 획기적으로 개선할 수 있습니다.

### [분석 2] 타겟(`clicked`) 기본 분포 및 시간축(`day_of_week`, `hour`) 구조 비교
- `train`의 전체 클릭률(Positive Rate)을 확인하여 `50:50 WLL` 최적화를 위한 클래스 불균형 비율(`N_0 / N_1`)을 산출합니다.
- `train`과 `test`의 `day_of_week` 및 `inventory_id` 분포를 대조하여, `test`가 미래 특정 요일(Out-of-Time)인지 동일 기간 분할인지 판별합니다.

In [ ]:
con = duckdb.connect()

target_stats = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(clicked) AS pos_rows,
        COUNT(*) - SUM(clicked) AS neg_rows,
        AVG(clicked) AS ctr
    FROM '{TRAIN_PATH.as_posix()}'
""").df()
display(target_stats)

train_dow = con.execute(f"""
    SELECT day_of_week, COUNT(*) AS train_cnt, AVG(clicked) AS train_ctr
    FROM '{TRAIN_PATH.as_posix()}'
    GROUP BY day_of_week
    ORDER BY day_of_week
""").df()

test_dow = con.execute(f"""
    SELECT day_of_week, COUNT(*) AS test_cnt
    FROM '{TEST_PATH.as_posix()}'
    GROUP BY day_of_week
    ORDER BY day_of_week
""").df()

dow_compare = pd.merge(train_dow, test_dow, on='day_of_week', how='outer').fillna(0)
display(dow_compare)

inv_stats = con.execute(f"""
    SELECT inventory_id, COUNT(*) AS train_cnt, AVG(clicked) AS train_ctr
    FROM '{TRAIN_PATH.as_posix()}'
    GROUP BY inventory_id
    ORDER BY train_cnt DESC
""").df()
test_inv = con.execute(f"""
    SELECT inventory_id, COUNT(*) AS test_cnt
    FROM '{TEST_PATH.as_posix()}'
    GROUP BY inventory_id
""").df()
inv_compare = pd.merge(inv_stats, test_inv, on='inventory_id', how='outer').fillna(0)
print(f'고유 inventory_id 수: {len(inv_compare)}, Test 미등장 신규 지면 수: {(inv_compare["train_cnt"] == 0).sum()}')
display(inv_compare.head(20))

### [해석 2] 클래스 불균형(`50:50 WLL`) 및 시간축 Local CV 전략 근거
- **타겟 불균형(`ctr`)**: 클릭률이 수 % 수준으로 낮기 때문에, 원천 확률 그대로 제출하면 양성(`clicked=1`) 로그손실이 폭증해 `50:50 WLL`이 크게 악화됩니다. 향후 학습 시 `scale_pos_weight = N_0 / N_1` 또는 Prior Shift(`shift_prob_prior`) 적용이 필수입니다.
- **시간축 구조(`day_of_week`)**: `test`에 `train`에 없는 미래 요일이 존재하거나 뒤쪽 요일에 분포가 편중되어 있다면 `is_oot_val=True`로 마지막 요일 홀드아웃 검증을 반드시 병행해야 하며, 동시에 전체 분포 안정을 위해 `(day_of_week, clicked)` 기준 5-Fold `StratifiedKFold`를 표준 CV로 고정합니다.
- **지면(`inventory_id`)**: 지면별 CTR 편차가 크고 신규/저빈도 지면 존재 여부에 따라 Out-of-Fold Target Encoding 시 Smoothing 강도를 조절해야 합니다.

### [분석 3] `seq` 시퀀스 컬럼 샘플 및 길이 분포 점검
- `seq` 컬럼의 실제 값 형태(예: 쉼표 구분 정수 시퀀스 등), 결측 여부, 문자열 길이/토큰 개수 분포를 20만 행 샘플로 빠르게 프로파일링합니다.

In [ ]:
seq_sample = con.execute(f"""
    SELECT seq, clicked
    FROM '{TRAIN_PATH.as_posix()}'
    LIMIT 5
""").df()
for i, row in seq_sample.iterrows():
    s = str(row['seq'])
    print(f'[{i}] clicked={row["clicked"]}, char_len={len(s)}, preview={s[:120]}...')

seq_len_stats = con.execute(f"""
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN seq IS NULL OR LENGTH(seq) = 0 THEN 1 ELSE 0 END) AS empty_cnt,
        AVG(LENGTH(seq)) AS avg_char_len,
        QUANTILE_CONT(LENGTH(seq), [0.1, 0.5, 0.9, 0.99, 1.0]) AS char_len_quantiles
    FROM (SELECT seq FROM '{TRAIN_PATH.as_posix()}' LIMIT 200000)
""").df()
display(seq_len_stats)

### [해석 3] `seq` 포맷 및 시퀀스 모델링/요약 피처 설계 근거
- 샘플 프리뷰로 구분자(쉼표 `,` 등)와 토큰 형태(정수 ID vs 범주 문자열)를 확인합니다.
- `char_len_quantiles`의 90~99% 백분위수는 향후 DeepCTR(`seq-deepctr-engineer`) 트랙에서 패딩/절단 최대 길이(`max_seq_len`)와 메모리 버퍼 크기를 결정하는 직접적인 기준이 됩니다.

### [분석 4] 정형 피처(117개) 결측률·고유값 수·범위(`min`/`max`) 요약 및 `eda_summary.json` 저장
- `seq`와 `clicked`를 제외한 117개 피처에 대해 결측률, 고유값 근사치(`APPROX_COUNT_DISTINCT`), 그리고 수치형 컬럼의 최솟값/최댓값(`MIN`/`MAX`)을 계산해 경량 JSON으로 저장합니다.
- 생성된 `eda_summary.json`은 `toss_ctr.data.schema`의 `load_eda_summary()` 및 `infer_safe_dtypes()`와 바로 호환됩니다.

In [ ]:
feature_cols = [c for c in train_pf.schema_arrow.names if c not in ('clicked', 'seq')]

numeric_cols = [
    c for c in feature_cols
    if pa.types.is_integer(train_pf.schema_arrow.field(c).type)
    or pa.types.is_floating(train_pf.schema_arrow.field(c).type)
]

select_exprs = ['COUNT(*) AS __total_rows']
for c in feature_cols:
    qc = f'"{c}"'
    select_exprs.append(f'COUNT({qc}) AS "{c}__non_null"')
    select_exprs.append(f'APPROX_COUNT_DISTINCT({qc}) AS "{c}__approx_nunique"')
    if c in numeric_cols:
        select_exprs.append(f'MIN({qc}) AS "{c}__min"')
        select_exprs.append(f'MAX({qc}) AS "{c}__max"')

profile_raw = con.execute(f"""
    SELECT {', '.join(select_exprs)}
    FROM '{TRAIN_PATH.as_posix()}'
""").df().iloc[0].to_dict()

def _clean_bound(val):
    if val is None or pd.isna(val):
        return None
    if isinstance(val, (np.integer, int)):
        return int(val)
    fval = float(val)
    return fval if math.isfinite(fval) else None

total_n = int(profile_raw['__total_rows'])
feat_profile = []
for c in feature_cols:
    non_null = int(profile_raw[f'{c}__non_null'])
    item = {
        'column': c,
        'arrow_type': str(train_pf.schema_arrow.field(c).type),
        'compressed_mb': round(float(col_sizes.get(c, 0) / (1024 ** 2)), 3),
        'null_rate': round(float(1.0 - non_null / total_n), 6),
        'approx_nunique': int(profile_raw[f'{c}__approx_nunique']),
    }
    if c in numeric_cols:
        item['min'] = _clean_bound(profile_raw.get(f'{c}__min'))
        item['max'] = _clean_bound(profile_raw.get(f'{c}__max'))
    feat_profile.append(item)

feat_profile_df = pd.DataFrame(feat_profile)
display(feat_profile_df.head(30))

# toss_ctr.data.schema가 설치되어 있다면 피처 그룹 분류 결과도 함께 기록합니다.
feature_groups = None
try:
    from toss_ctr.data.schema import discover_feature_groups
    feature_groups = discover_feature_groups(feature_cols)
    print('피처 그룹별 컬럼 수:', {k: len(v) for k, v in feature_groups.items()})
except ImportError:
    pass

summary_payload = {
    'train_rows': int(train_pf.metadata.num_rows),
    'test_rows': int(test_pf.metadata.num_rows),
    'target_stats': target_stats.to_dict(orient='records')[0],
    'day_of_week_compare': dow_compare.to_dict(orient='records'),
    'inventory_top50': inv_compare.head(50).to_dict(orient='records'),
    'seq_compressed_mb': round(float(col_sizes.get('seq', 0) / (1024 ** 2)), 2),
    'feature_groups': feature_groups,
    'features': feat_profile,
}

summary_path = OUT_DIR / 'eda_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary_payload, f, ensure_ascii=False, indent=2)

print(f'Saved lightweight summary to: {summary_path} ({summary_path.stat().st_size / 1024:.1f} KB)')

### [해석 4] 정형 피처 프로파일 및 `eda_summary.json` 저장 결과
- 수치형 컬럼의 `min`/`max`와 `null_rate`, `approx_nunique`가 모두 `eda_summary.json`에 기록되었습니다.
- `null_rate == 0`이고 범위가 좁은 정수 컬럼은 `int8`/`int16`으로, 고밀도 실수 컬럼은 `float32`로 안전하게 다운캐스팅할 수 있음을 사전에 확인할 수 있습니다.

### [결론 및 다음 단계 체크리스트]
- 위 실행으로 생성된 `eda_summary.json`을 확인하고(필요 시 로컬 `reports/eda_summary.json`으로 백업), 다음 단계인 **`notebooks/02_data_diet_and_cv.ipynb`**를 실행해 주세요.
- `02_data_diet_and_cv.ipynb`에서는 `toss_ctr.data.diet.build_diet_artifacts()`와 `toss_ctr.data.splits.build_cv_folds()`를 통해 **OOM 없는 경량 Parquet(`train_tabular_diet`, `test_tabular_diet`, `train_seq`, `test_seq`) 및 공통 `folds.parquet`**을 확정합니다.